# Data Loading Notebook

This notebook demonstrates how to load different versions of the hedge fund dataset:
- Full train/test data (if available)
- Sample train/test data (for GitHub users)
- Cached data loading for performance

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Define Data Paths

In [ ]:
# Define base directory
base_dir = Path("..")  # Go up one level from notebooks to project root

# Full data paths
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

# Sample data paths
sample_train_path = base_dir / "data" / "sample" / "train_sample.parquet"
sample_test_path = base_dir / "data" / "sample" / "test_sample.parquet"

print(f"Full train data: {full_train_path}")
print(f"Full test data: {full_test_path}")
print(f"Sample train data: {sample_train_path}")
print(f"Sample test data: {sample_test_path}")

## 2. Check Available Data

In [1]:
def check_data_availability():
    """Check which data files are available"""
    data_status = {
        "full_train": full_train_path.exists(),
        "full_test": full_test_path.exists(),
        "sample_train": sample_train_path.exists(),
        "sample_test": sample_test_path.exists()
    }
    
    print("Data Availability:")
    print("=" * 40)
    for name, available in data_status.items():
        status = "✅ Available" if available else "❌ Missing"
        print(f"{name}: {status}")
    
    return data_status

data_status = check_data_availability()

NameError: name 'full_train_path' is not defined

## 3. Load Sample Data (Recommended for GitHub users)

In [ ]:
def load_sample_data():
    """Load sample data for quick analysis"""
    print("Loading sample data...")
    
    # Load sample train data
    if sample_train_path.exists():
        train_df = pd.read_parquet(sample_train_path)
        print(f"✅ Sample train data loaded: {train_df.shape}")
    else:
        print("❌ Sample train data not found")
        train_df = None
    
    # Load sample test data
    if sample_test_path.exists():
        test_df = pd.read_parquet(sample_test_path)
        print(f"✅ Sample test data loaded: {test_df.shape}")
    else:
        print("❌ Sample test data not found")
        test_df = None
    
    return train_df, test_df

sample_train_df, sample_test_df = load_sample_data()

## 4. Load Full Data (If available)

In [ ]:
def load_full_data():
    """Load full dataset (may take time)"""
    print("Loading full data (this may take a while)...")
    
    # Load full train data
    if full_train_path.exists():
        print("Loading full train data...")
        train_df = pd.read_parquet(full_train_path)
        print(f"✅ Full train data loaded: {train_df.shape}")
    else:
        print("❌ Full train data not found")
        train_df = None
    
    # Load full test data
    if full_test_path.exists():
        print("Loading full test data...")
        test_df = pd.read_parquet(full_test_path)
        print(f"✅ Full test data loaded: {test_df.shape}")
    else:
        print("❌ Full test data not found")
        test_df = None
    
    return train_df, test_df

# Uncomment to load full data
# full_train_df, full_test_df = load_full_data()

## 5. Data Overview

In [ ]:
def show_data_overview(df, data_name="Data"):
    """Display basic information about the dataset"""
    if df is None:
        print(f"❌ {data_name} not available")
        return
    
    print(f"\n{data_name} Overview:")
    print("=" * 50)
    print(f"Shape: {df.shape}")
    print(f"Columns: {len(df.columns)}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    print(f"Missing values: {df.isnull().sum().sum()}")
    
    print("\nFirst few columns:")
    print(df.columns[:10].tolist())
    
    print("\nData types:")
    print(df.dtypes.value_counts())
    
    print("\nSample of data:")
    display(df.head(3))

# Show sample data overview
if sample_train_df is not None:
    show_data_overview(sample_train_df, "Sample Train Data")

if sample_test_df is not None:
    show_data_overview(sample_test_df, "Sample Test Data")

## 6. Target Variable Analysis

In [ ]:
def analyze_target_variable(df, data_name="Data"):
    """Analyze the target variable"""
    if df is None or 'y_target' not in df.columns:
        print(f"❌ Target variable not found in {data_name}")
        return
    
    target = df['y_target']
    
    print(f"\n{data_name} - Target Variable Analysis:")
    print("=" * 50)
    print(f"Mean: {target.mean():.6f}")
    print(f"Std: {target.std():.6f}")
    print(f"Min: {target.min():.6f}")
    print(f"Max: {target.max():.6f}")
    print(f"Missing: {target.isnull().sum()}")
    
    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f'{data_name} - Target Distribution', fontsize=16)
    
    # Histogram
    axes[0, 0].hist(target.dropna(), bins=50, alpha=0.7, color='steelblue')
    axes[0, 0].set_title('Histogram')
    axes[0, 0].set_xlabel('y_target')
    axes[0, 0].set_ylabel('Frequency')
    
    # Boxplot
    axes[0, 1].boxplot(target.dropna())
    axes[0, 1].set_title('Boxplot')
    axes[0, 1].set_ylabel('y_target')
    
    # Density plot
    target.plot(kind='density', ax=axes[1, 0], color='steelblue')
    axes[1, 0].set_title('Density Plot')
    axes[1, 0].set_xlabel('y_target')
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(target.dropna(), dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title('Q-Q Plot')
    
    plt.tight_layout()
    plt.show()

# Analyze sample train data
if sample_train_df is not None:
    analyze_target_variable(sample_train_df, "Sample Train Data")

## 7. Feature Analysis

In [ ]:
def analyze_features(df, data_name="Data"):
    """Analyze feature characteristics"""
    if df is None:
        print(f"❌ {data_name} not available")
        return
    
    print(f"\n{data_name} - Feature Analysis:")
    print("=" * 50)
    
    # Feature types
    numeric_features = df.select_dtypes(include=[np.number]).columns
    categorical_features = df.select_dtypes(include=['object', 'category']).columns
    
    print(f"Numeric features: {len(numeric_features)}")
    print(f"Categorical features: {len(categorical_features)}")
    
    # Missing values by feature
    missing_values = df.isnull().sum()
    missing_features = missing_values[missing_values > 0]
    
    if len(missing_features) > 0:
        print(f"\nFeatures with missing values ({len(missing_features)}):")
        print(missing_features.head(10))
    else:
        print("\n✅ No missing values found")
    
    # Correlation matrix for numeric features (sample)
    if len(numeric_features) > 0:
        # Sample of correlations to avoid memory issues
        sample_numeric = df[numeric_features[:20]]  # First 20 numeric features
        correlations = sample_numeric.corr()
        
        # Plot correlation heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlations, annot=False, cmap='coolwarm', center=0)
        plt.title(f'{data_name} - Feature Correlations (Sample)')
        plt.tight_layout()
        plt.show()

# Analyze sample train data features
if sample_train_df is not None:
    analyze_features(sample_train_df, "Sample Train Data")

## 8. Data Loading Summary

In [ ]:
def show_loading_summary():
    """Show summary of available data and recommendations"""
    print("\n" + "=" * 60)
    print("DATA LOADING SUMMARY")
    print("=" * 60)
    
    # Check what's available
    has_sample_data = sample_train_df is not None and sample_test_df is not None
    has_full_data = full_train_path.exists() and full_test_path.exists()
    
    print(f"\n✅ Sample Data Available: {has_sample_data}")
    print(f"✅ Full Data Available: {has_full_data}")
    
    if has_sample_data:
        print(f"\n📊 Sample Data Sizes:")
        print(f"   Train: {sample_train_df.shape}")
        print(f"   Test: {sample_test_df.shape}")
    
    print(f"\n💡 Recommendations:")
    if has_sample_data:
        print("   • Use sample data for development and testing")
        print("   • Use sample data for quick prototyping")
    if has_full_data:
        print("   • Use full data for final model training")
        print("   • Consider caching for repeated experiments")
    
    print(f"\n📁 Data Paths:")
    print(f"   Sample train: {sample_train_path}")
    print(f"   Sample test: {sample_test_path}")
    print(f"   Full train: {full_train_path}")
    print(f"   Full test: {full_test_path}")

show_loading_summary()

## 9. Save Processed Data (Optional)

In [ ]:
def save_processed_data(df, filename, data_type="processed"):
    """Save processed data for later use"""
    if df is None:
        print(f"❌ No data to save")
        return
    
    output_dir = base_dir / "data" / data_type
    output_dir.mkdir(exist_ok=True)
    
    output_path = output_dir / filename
    df.to_parquet(output_path, index=False)
    
    print(f"✅ Data saved to: {output_path}")
    print(f"   Size: {output_path.stat().st_size / 1024**2:.1f} MB")

# Example usage (uncomment to save):
# if sample_train_df is not None:
#     save_processed_data(sample_train_df, "train_processed.parquet", "processed")
# if sample_test_df is not None:
#     save_processed_data(sample_test_df, "test_processed.parquet", "processed")